Auteur:     Eva Rombouts  
Datum:      23-12-2023  
Project:    GenCareAI  
Doel:       Verkennen en opschonen van de data die met de OpenAI API is gegenereerd.

In het 010 script zijn clientprofielen gemaakt voor 24 clienten met een vorm van dementie. De profielen zijn verhalend en ongestructureerd. Dit heb ik zo gelaten omdat het in de praktijk ook vaak zo gebeurt. 
In het 015 script zijn voor deze 24 clienten 10 weken aan rapportages gegenereerd. In deze rapportages is wel een vaste structuur aangebracht, maar het blijft nog een beetje spannend of die overal goed is gevolgd en consistent is. 
In dit script gaan we de opgeslagen json file omzetten naar twee datasets: df_clienten en df_rapportages.

Modellen:  
MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (multi NLI)
wietsedv/bert-base-dutch-cased: Gebruikt door de michelin jongens. Op de site <mask>
Robbert: pdelobelle/robbert-v2-dutch-base

In [2]:
# Setup
import json
import re
import pandas as pd

# Parameters
seed = 6
filename_rapportages = '../zorgdata/appelboom_rapportages.json' # Bevat de opgeslagen JSON file
filename_clienten = '../zorgdata/appelboom_clienten.json' # Hier wordt de gecleande data naar geschreven

In [3]:
# Lees de clientendata in
with open(filename_rapportages, 'r') as file:
    appelboom_rapportages = json.load(file)

De structuur van de appelboom_rapportages dictionary kan worden verbeterd en opgeschoond. 

Voor elke cliënt in appelboom_rapportages wordt een unieke ID gegenereerd in de vorm van kamer01, kamer02, enzovoorts. Deze unieke ID's vervangen de oorspronkelijke keys die beginnen met 'Naam: '.

De originele keys beginnen allemaal met 'Naam: '. Deze prefix wordt verwijderd om alleen de daadwerkelijke naam van de cliënt over te houden. Indien de naam 'Meneer' of 'Mevrouw' bevat, wordt dit gebruikt om het geslacht van de cliënt te bepalen. De naam wordt vervolgens opgeschoond door deze termen te verwijderen. Als de naam geen van deze aanduidingen bevat, wordt het geslacht als 'onbekend' gemarkeerd.

Voor elke cliënt worden de gegevens opnieuw gestructureerd. De 'Profiel' informatie wordt apart opgeslagen en alle rapportages die beginnen met 'Week' worden samengevoegd onder een aparte key 'rapportages'.

Het resultaat is een opgeschoonde en gestructureerde dictionary appelboom_clienten, waarbij elke cliënt een uniek ID heeft (het kamernummer) en de gegevens overzichtelijk zijn georganiseerd. 

In [4]:
appelboom_clienten = {}

for i, (key, value) in enumerate(appelboom_rapportages.items()):
    uniek_id = f"kamer{i+1:02d}"  # Genereert ID's zoals 'kamer01', 'kamer02', etc.
    naam = key.replace('Naam: ', '')
    naam_geslacht = naam.split(' ', 1)
    if 'Meneer' in naam_geslacht[0]:
        geslacht = 'man' 
        naam = naam_geslacht[1]
    elif 'Mevrouw' in naam_geslacht[0]:
        geslacht = 'vrouw' 
        naam = naam_geslacht[1]
    else:
        geslacht = 'onbekend'
    
    appelboom_clienten[uniek_id] = {
        "naam": naam,
        "geslacht": geslacht,
        "profiel": value.get("Profiel", ""),
        "rapportages": {k: v for k, v in value.items() if k.startswith("Week")}
    }


In [5]:
# Schrijf weg. 
with open(filename_clienten, 'w') as file:
    json.dump(appelboom_clienten, file) 

In [9]:
# Voorbeeldje
profiel = appelboom_clienten['kamer01']['profiel']
print(profiel)

Naam: Meneer Pietersen

Dementietype: Alzheimer

Lichamelijke klachten:
- Meneer Pietersen heeft last van artritis, waardoor hij moeite heeft met bewegen en dagelijkse taken uitvoeren, zoals zichzelf aankleden en eten.
- Hij heeft ook last van gehoorverlies, wat kan leiden tot miscommunicatie en het moeilijk begrijpen van gesprekken.
- Daarnaast heeft hij een hoge bloeddruk en diabetes, waardoor er regelmatig bloeddrukcontroles en medicatiebeheer nodig zijn.

Beschrijving:
Meneer Pietersen is een vriendelijke en zachtaardige man van 78 jaar oud. Hij was vroeger bankmanager en heeft altijd een scherp geheugen gehad. Helaas begon hij een paar jaar geleden symptomen van Alzheimer te vertonen, wat leidde tot geheugenverlies en verwarring.

Vanwege zijn fysieke klachten heeft meneer Pietersen behoefte aan ondersteuning bij dagelijkse activiteiten, zoals douchen, omkleden en eten. Hij kan soms verward raken en verdwaald raken in zijn eigen vertrouwde omgeving, dus toezicht en begeleiding zij

Je ziet dat de gegenereerde data lekker biased is. Meneer Pieterse was bankmanager. Als ik snel kijk zijn er ook twee architecten, allebei man. De dames zijn artistiek, doen vrijwilligerswerk, zijn familiemensen, op waren lerares...
Maar... het is wel realistisch.

In [79]:
# voorbeeld rapportage_tekst
rapportage_tekst = appelboom_clienten['kamer01']['rapportages']['Week 2']
print(rapportage_tekst)

---StartRapportage---
Dag: Maandag
Niveau: Verzorgende
Rapportage: Vanochtend heeft meneer Pietersen moeite gehad met opstaan uit bed vanwege zijn artritis. Ik heb hem geholpen met een pijnstiller en hem ondersteund bij het aankleden. Tijdens het ontbijt raakte hij verward en kon hij niet goed vinden wat hij wilde eten. Ik heb hem geholpen bij het maken van een keuze en heb hem rustig uitgelegd wat er op zijn bord lag. Daarna heeft hij rustig gegeten. In de middag hebben we samen een puzzel gemaakt, wat hem hielp om te ontspannen. Hij was vandaag over het algemeen wat onrustig en verward, dus heb ik extra in de gaten gehouden of hij niet verdwaalde in huis.
Onrustscore: 60
--EindRapportage---

---StartRapportage---
Dag: Dinsdag
Niveau: Helpende
Rapportage: Gisterenmiddag hebben we meneer Pietersen meegenomen voor een wandeling in de tuin. Hij vond het fijn om buiten te zijn en genoot van de frisse lucht. Na het avondeten merkte ik dat hij wat rusteloos was en moeite had om stil te zitt

### Parsen van weekrapportages
Bij het genereren van de data is het llm gevraagd om rapportages van een week in gestructureerde vorm te maken. 
Deze worden gestart met ---StartRapportage--- en eindigen met --EindRapportage--- (let op. Per abuis start dit met 2 streepjes). 
vervolgens zijn dag/niveau/rapportage en onrustscore vaste kopjes. De onrustscore zit vaak zonder newline aan de rapportage geplakt. 

Onderstaande functie parst de weekrapportages en geeft een list met dictionaries terug.

In [81]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


In [82]:
aap = split_rapportage(rapportage_tekst)
# print (rapportage_tekst)

for ap in aap:
    print('-'*100)
    print (ap)

----------------------------------------------------------------------------------------------------
Dag: Maandag
Niveau: Verzorgende
Rapportage: Vanochtend heeft meneer Pietersen moeite gehad met opstaan uit bed vanwege zijn artritis. Ik heb hem geholpen met een pijnstiller en hem ondersteund bij het aankleden. Tijdens het ontbijt raakte hij verward en kon hij niet goed vinden wat hij wilde eten. Ik heb hem geholpen bij het maken van een keuze en heb hem rustig uitgelegd wat er op zijn bord lag. Daarna heeft hij rustig gegeten. In de middag hebben we samen een puzzel gemaakt, wat hem hielp om te ontspannen. Hij was vandaag over het algemeen wat onrustig en verward, dus heb ik extra in de gaten gehouden of hij niet verdwaalde in huis.
Onrustscore: 60
----------------------------------------------------------------------------------------------------
Dag: Dinsdag
Niveau: Helpende
Rapportage: Gisterenmiddag hebben we meneer Pietersen meegenomen voor een wandeling in de tuin. Hij vond het

In [15]:
# Werkt toch niet helemaal goed... Ik probeer het nog 's. 
def parse_weekrapportages(rapportage_tekst):
    # Stap 1: Splits de tekst in individuele rapportages
#    rapportages = re.split(r'---StartRapportage---|--EindRapportage---', rapportage_tekst)
    rapportages = re.split(r'-+\s*StartRapportage\s-+|-+\s*EindRapportage\s-+', rapportage_tekst)
    rapportages = [r.strip() for r in rapportages if r.strip()]

    # Stap 2: Extraheer informatie uit elke rapportage
    parsed_rapportages = []
    for rapport in rapportages:
        # Splits de rapportage op 'Onrustscore'
        delen = rapport.split('Onrustscore:')
        rapportage = delen[0].strip()

        onrustscore_match = re.search(r'Onrustscore:\s*(\d+)', rapport)
        onrustscore = int(onrustscore_match.group(1)) if onrustscore_match else None

#        onrustscore = int(delen[1].strip()) if len(delen) > 1 else None

        # Extraheer dag en niveau
        dag = re.search(r'Dag: (.+)', rapportage).group(1)
        niveau = re.search(r'Niveau: (.+)', rapportage).group(1)

        # Verwijder dag en niveau uit de rapportagetekst
        rapportage = re.sub(r'Dag: .+\n', '', rapportage)
        rapportage = re.sub(r'Niveau: .+\n', '', rapportage)

        parsed_rapportages.append({
            "dag": dag,
            "niveau": niveau,
            "rapportage": rapportage,
            "onrustscore": onrustscore
        })

    return parsed_rapportages


In [17]:
def parse_weekrapportages(rapportage_list):

    # Stap 2: Extraheer informatie uit elke rapportage
    parsed_rapportages = []
    for rapport in rapportages:
        # Splits de rapportage op 'Onrustscore'
        delen = rapport.split('Onrustscore:')
        rapportage = delen[0].strip()

        onrustscore_match = re.search(r'Onrustscore:\s*(\d+)', rapport)
        onrustscore = int(onrustscore_match.group(1)) if onrustscore_match else None

#        onrustscore = int(delen[1].strip()) if len(delen) > 1 else None

        # Extraheer dag en niveau
        dag = re.search(r'Dag: (.+)', rapportage).group(1)
        niveau = re.search(r'Niveau: (.+)', rapportage).group(1)

        # Verwijder dag en niveau uit de rapportagetekst
        rapportage = re.sub(r'Dag: .+\n', '', rapportage)
        rapportage = re.sub(r'Niveau: .+\n', '', rapportage)

        parsed_rapportages.append({
            "dag": dag,
            "niveau": niveau,
            "rapportage": rapportage,
            "onrustscore": onrustscore
        })

    return parsed_rapportages


['---StartRapportage---\nDag: Maandag\nNiveau: Helpende\nRapportage: Vandaag heb ik meneer Pietersen geholpen met het aankleden en het wassen. Hij had wat moeite met het bewegen van zijn armen vanwege zijn artritis, maar ik heb hem rustig geholpen en hij leek zich comfortabel te voelen. Hij was wat verward tijdens het ontbijt en vroeg me herhaaldelijk waar hij was, maar ik heb hem gerustgesteld en hem verteld dat hij in het verpleeghuis was. In de middag hebben we samen een puzzel gemaakt en dat leek hem rustig te maken. Hij vertoonde weinig onrust vandaag.\nOnrustscore: 30\n--EindRapportage---\n\n---StartRapportage---\nDag: Dinsdag\nNiveau: Verzorgende\nRapportage: Meneer Pietersen was vanochtend erg onrustig. Hij was boos en verward en wilde niet meewerken bij het aankleden. Toen ik zag dat hij geïrriteerd was, heb ik hem wat tijd gegeven om te kalmeren en zijn stemming verbeterde iets. Tijdens het ontbijt had hij moeite met het vasthouden van zijn bestek, vanwege zijn artritis. Ik h

In [33]:
import pprint

aap = parse_weekrapportages(rapportage_tekst)
pprint.pprint(aap)


[{'dag': 'Maandag',
  'niveau': 'Verzorgende',
  'onrustscore': 15,
  'rapportage': '---StartRapportage---\n'
                'Rapportage: Vandaag had Albert wat moeite met opstaan en '
                'zichzelf aankleden vanwege zijn parkinsonisme. Ik heb hem '
                'geholpen met het aantrekken van zijn shirt en broek, en heb '
                'ervoor gezorgd dat hij zijn medicijnen heeft ingenomen. '
                'Daarna hebben we samen wat lichte oefeningen gedaan om zijn '
                'mobiliteit te verbeteren. Albert was over het algemeen rustig '
                'en meewerkend vandaag.'}]


In [ ]:
for rapport in aap:
    print(f"Dag: {rapport['dag']}")
    print(f"Niveau: {rapport['niveau']}")
    print("Rapportage:")
    print(rapport['rapportage'])
    print(f"Onrustscore: {rapport['onrustscore']}")
    print("-" * 40)  # Print een scheidingslijn


In [ ]:
df = pd.DataFrame(aap)
print(df)

In [8]:
# Maak een DataFrame voor de cliënten
df_clienten = pd.DataFrame.from_dict(appelboom_clienten, orient='index')
df_clienten.reset_index(inplace=True)
df_clienten.rename(columns={'index': 'kamernummer'}, inplace=True)
df_clienten.drop(columns=['rapportages'], inplace=True)


In [ ]:
print(df_clienten)

In [34]:
# Verzamelen van alle geparseerde rapportages in een lijst
parsed_rapportages = []
for kamernummer, gegevens in appelboom_clienten.items():
    for weeknummer, rapportage_tekst in gegevens['rapportages'].items():
        rapportages = parse_weekrapportages(rapportage_tekst)
        for rapport in rapportages:
            rapport['kamernummer'] = kamernummer
            rapport['weeknummer'] = weeknummer
            parsed_rapportages.append(rapport)

# Maak een DataFrame voor de rapportages
df_rapportages = pd.DataFrame(parsed_rapportages)


In [ ]:
print(df_rapportages)